# Stories Coffee — Step 1: data audit

**Question:** Can we use this file reliably, and which limitations must we carry into our analysis?

This notebook checks the data before we choose a business recommendation. It does not train a model or prove that any business action will work.

**On Zahraa's PC:** open this notebook in VS Code, select the Python environment where the requirements are installed, and click **Run All**. Put the original CSV in `data/raw/stories_transactions.csv`. The supplied starter already includes it.

The original CSV stays unchanged. Parsed dates and other working values exist only in a separate dataframe. Duplicate removal is a sensitivity scenario, not a cleaning decision.

The saved outputs were produced by executing these code cells in order with Python. You can rerun them in a Jupyter kernel in VS Code.


## 1. Load the original file

The SHA-256 fingerprint lets us check that we are working with exactly the same bytes. It does not establish that the source data is complete or correct.


In [1]:
from pathlib import Path
import csv
import hashlib
import json
import platform
import pandas as pd

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)

# This works when VS Code starts in the project folder or the notebooks folder.
candidates = [
    Path.cwd() / 'data/raw/stories_transactions.csv',
    Path.cwd().parent / 'data/raw/stories_transactions.csv',
]
data_path = next((p.resolve() for p in candidates if p.is_file()), None)
if data_path is None:
    raise FileNotFoundError('Place stories_transactions.csv in data/raw, then rerun from the project or notebooks folder.')
project_root = data_path.parents[2]
source_hash = hashlib.sha256(data_path.read_bytes()).hexdigest()

with data_path.open(encoding='utf-8-sig', newline='') as f:
    header = next(csv.reader(f))
if len(header) != len(set(header)):
    raise ValueError('The CSV has repeated column names; investigate before continuing.')

raw = pd.read_csv(data_path, dtype={'Transaction ID': 'string', 'Customer ID': 'string'})
required_columns = [
    'Transaction ID', 'Date', 'Time', 'Branch', 'Customer ID', 'Order Type',
    'Product', 'Category', 'Quantity', 'Unit Price (USD)', 'Discount Rate',
    'Unit Cost (USD)', 'Line Revenue (USD)', 'Line Gross Profit (USD)',
]
if set(raw.columns) != set(required_columns):
    raise ValueError('The column names differ from the supplied file. Review the schema.')

print('Rows:', f'{len(raw):,}', '| Columns:', len(raw.columns))
print('Each row is a product line. A purchase may contain several rows.')
print('SHA-256:', source_hash)
print('Environment:', platform.python_version(), '| pandas', pd.__version__)


Rows: 51,177 | Columns: 14
Each row is a product line. A purchase may contain several rows.
SHA-256: f1672e728c28295b2b802d878fd38a158ba039827b429355e8438ee0463237a1
Environment: 3.13.7 | pandas 2.2.3


## 2. Check missing values and text formatting

We do not fill missing customer IDs with one shared ID: that would incorrectly turn many unidentified people into a single customer. Missing IDs do not prevent branch-level sales analysis, but they limit customer-level conclusions.


In [2]:
missing = pd.DataFrame({
    'missing_rows': raw.isna().sum(),
    'missing_percent': (raw.isna().mean() * 100).round(2),
})
print(missing.to_string())

text_columns = ['Transaction ID', 'Date', 'Time', 'Branch', 'Customer ID', 'Order Type', 'Product', 'Category']
text_issues = {}
for column in text_columns:
    values = raw[column].astype('string')
    text_issues[column] = int((values.notna() & (values != values.str.strip())).sum())
print('Rows with leading/trailing spaces, by text column:', text_issues)
print('No text values have been changed.')


                         missing_rows  missing_percent
Transaction ID                      0             0.00
Date                                0             0.00
Time                                0             0.00
Branch                              0             0.00
Customer ID                     18691            36.52
Order Type                          0             0.00
Product                             0             0.00
Category                            0             0.00
Quantity                            0             0.00
Unit Price (USD)                    0             0.00
Discount Rate                       0             0.00
Unit Cost (USD)                     0             0.00
Line Revenue (USD)                  0             0.00
Line Gross Profit (USD)             0             0.00
Rows with leading/trailing spaces, by text column: {'Transaction ID': 0, 'Date': 0, 'Time': 0, 'Branch': 0, 'Customer ID': 0, 'Order Type': 0, 'Product': 0, 'Category': 0}
No 

## 3. Check dates, times, and numeric fields

We parse dates and numbers in a working copy. Negative gross profit is reported separately because a genuine loss is a possible business outcome, not automatically a data error.


In [3]:
df = raw.copy()
numeric_columns = ['Quantity', 'Unit Price (USD)', 'Discount Rate', 'Unit Cost (USD)', 'Line Revenue (USD)', 'Line Gross Profit (USD)']
checks = {}
for column in numeric_columns:
    df[column] = pd.to_numeric(raw[column], errors='coerce')
    checks['Missing or non-numeric: ' + column] = int(df[column].isna().sum())

df['parsed_date'] = pd.to_datetime(raw['Date'], format='%Y-%m-%d', errors='coerce')
df['timestamp'] = pd.to_datetime(raw['Date'] + ' ' + raw['Time'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
checks['Invalid or missing dates'] = int(df['parsed_date'].isna().sum())
checks['Invalid or missing timestamps'] = int(df['timestamp'].isna().sum())
checks['Missing required non-customer fields'] = int(raw.drop(columns='Customer ID').isna().sum().sum())
checks['Non-positive or non-integer quantities'] = int(((df['Quantity'] <= 0) | (df['Quantity'] % 1 != 0)).sum())
checks['Negative unit prices'] = int((df['Unit Price (USD)'] < 0).sum())
checks['Negative unit costs'] = int((df['Unit Cost (USD)'] < 0).sum())
checks['Discounts outside 0 to 1'] = int((~df['Discount Rate'].between(0, 1)).sum())

print(pd.Series(checks, name='flagged_count').to_string())
print('Observed quantities:', sorted(df['Quantity'].unique().tolist()))
print('Observed discount rates:', sorted(df['Discount Rate'].unique().tolist()))
print('Negative revenue rows:', int((df['Line Revenue (USD)'] < 0).sum()))
print('Negative gross-profit rows:', int((df['Line Gross Profit (USD)'] < 0).sum()))
if any(checks.values()):
    raise ValueError('A structural check failed. Investigate before later analysis.')


Missing or non-numeric: Quantity                   0
Missing or non-numeric: Unit Price (USD)           0
Missing or non-numeric: Discount Rate              0
Missing or non-numeric: Unit Cost (USD)            0
Missing or non-numeric: Line Revenue (USD)         0
Missing or non-numeric: Line Gross Profit (USD)    0
Invalid or missing dates                           0
Invalid or missing timestamps                      0
Missing required non-customer fields               0
Non-positive or non-integer quantities             0
Negative unit prices                               0
Negative unit costs                                0
Discounts outside 0 to 1                           0
Observed quantities: [1, 2]
Observed discount rates: [0.0, 0.1, 0.15]
Negative revenue rows: 0
Negative gross-profit rows: 0


## 4. Verify the financial calculations

Expected line revenue = quantity × unit price × (1 − discount).

Expected line gross profit = recorded line revenue − quantity × unit cost.

We allow one cent per line for rounding, with a tiny numerical tolerance. Passing these checks establishes arithmetic consistency; it does not verify supplier invoices, cost allocation, or net profit. Rent, wages, and other expenses are not separately provided.


In [4]:
expected_revenue = df['Quantity'] * df['Unit Price (USD)'] * (1 - df['Discount Rate'])
expected_gross_profit = df['Line Revenue (USD)'] - df['Quantity'] * df['Unit Cost (USD)']
revenue_error = (df['Line Revenue (USD)'] - expected_revenue).abs()
profit_error = (df['Line Gross Profit (USD)'] - expected_gross_profit).abs()
money_tolerance = 0.01000001
checks['Revenue mismatches over one cent'] = int((revenue_error > money_tolerance).sum())
checks['Gross-profit mismatches over one cent'] = int((profit_error > money_tolerance).sum())
print('Revenue mismatches:', checks['Revenue mismatches over one cent'])
print('Gross-profit mismatches:', checks['Gross-profit mismatches over one cent'])
print('Maximum absolute revenue error:', round(float(revenue_error.max()), 8), 'USD')
print('Maximum absolute gross-profit error:', round(float(profit_error.max()), 8), 'USD')
if checks['Revenue mismatches over one cent'] or checks['Gross-profit mismatches over one cent']:
    raise ValueError('Financial arithmetic needs investigation.')


Revenue mismatches: 0
Gross-profit mismatches: 0
Maximum absolute revenue error: 0.005 USD
Maximum absolute gross-profit error: 0.0 USD


## 5. Reconstruct purchases and check consistency

Transaction IDs are replacement identifiers supplied with the data. We assume each distinct ID identifies one purchase. All lines within that purchase should agree on its date, time, branch, order type, and customer ID. We do not infer time order from the numeric part of the ID.


In [5]:
metadata_columns = ['Date', 'Time', 'Branch', 'Order Type', 'Customer ID']
distinct_values = raw.groupby('Transaction ID')[metadata_columns].nunique(dropna=False)
metadata_conflicts = (distinct_values > 1).sum()
print('Transactions with conflicting metadata:')
print(metadata_conflicts.to_string())
checks['Transactions with any metadata conflict'] = int((distinct_values > 1).any(axis=1).sum())
if checks['Transactions with any metadata conflict']:
    raise ValueError('Some transaction IDs have conflicting metadata; resolve before aggregation.')

transactions = df.groupby('Transaction ID').agg(
    date=('parsed_date', 'first'),
    timestamp=('timestamp', 'first'),
    branch=('Branch', 'first'),
    order_type=('Order Type', 'first'),
    customer_id=('Customer ID', 'first'),
    line_count=('Product', 'size'),
    units=('Quantity', 'sum'),
    revenue_usd=('Line Revenue (USD)', 'sum'),
    gross_profit_usd=('Line Gross Profit (USD)', 'sum'),
)
missing_customer_purchases = int(transactions['customer_id'].isna().sum())
print('Distinct purchases:', f'{len(transactions):,}')
print('Recorded customer IDs:', f'{raw["Customer ID"].nunique():,}')
print('Purchases without a customer ID:', f'{missing_customer_purchases:,}',
      f'({missing_customer_purchases / len(transactions):.2%})')
print('Product-line counts per purchase:')
print(transactions['line_count'].value_counts().sort_index().to_string())
print('Known IDs are not a count of all customers. We do not assume missing IDs are random.')


Transactions with conflicting metadata:
Date           0
Time           0
Branch         0
Order Type     0
Customer ID    0


Distinct purchases: 33,887
Recorded customer IDs: 3,494
Purchases without a customer ID: 12,353 (36.45%)
Product-line counts per purchase:
line_count
1    18175
2    14134
3     1578
Known IDs are not a count of all customers. We do not assume missing IDs are random.


## 6. Investigate repeated rows without deleting them

An exact repeat includes the same transaction ID and every other field. It could be an accidental duplicate, or two separate identical items in one basket. There is no line-item ID to distinguish these explanations.

**Primary decision:** retain every row. **Sensitivity scenario:** remove only exact repeats, then compare totals. This scenario does not establish which version is correct. A small overall difference may still matter in a small subgroup.


In [6]:
repeat_mask = raw.duplicated(keep='first')
affected_mask = raw.duplicated(keep=False)
repeat_count = int(repeat_mask.sum())
repeat_transactions = int(raw.loc[affected_mask, 'Transaction ID'].nunique())
sensitivity_only = df.loc[~repeat_mask].copy()

def totals(frame):
    return {
        'Product lines': len(frame),
        'Purchases': int(frame['Transaction ID'].nunique()),
        'Units': int(frame['Quantity'].sum()),
        'Revenue USD': round(float(frame['Line Revenue (USD)'].sum()), 2),
        'Gross profit USD': round(float(frame['Line Gross Profit (USD)'].sum()), 2),
    }

primary_totals = totals(df)
sensitivity_totals = totals(sensitivity_only)
sensitivity = pd.DataFrame({'Keep all rows (primary)': primary_totals,
                            'Remove exact repeats (scenario)': sensitivity_totals})
sensitivity['Scenario minus primary'] = sensitivity.iloc[:, 1] - sensitivity.iloc[:, 0]
print('Extra identical rows:', repeat_count)
print('All rows involved in exact repeats:', int(affected_mask.sum()))
print('Affected purchases:', repeat_transactions)
print(sensitivity.to_string(float_format=lambda value: f'{value:,.2f}'))

duplicate_profit = round(primary_totals['Gross profit USD'] - sensitivity_totals['Gross profit USD'], 2)
duplicate_profit_percent = 100 * duplicate_profit / primary_totals['Gross profit USD']
duplicate_revenue = round(primary_totals['Revenue USD'] - sensitivity_totals['Revenue USD'], 2)
print(f'Removing exact repeats reduces total gross profit by ${duplicate_profit:,.2f} ({duplicate_profit_percent:.3f}%).')

primary_branch = df.groupby('Branch').agg(revenue_usd=('Line Revenue (USD)', 'sum'), gross_profit_usd=('Line Gross Profit (USD)', 'sum'))
removed_branch = df.loc[repeat_mask].groupby('Branch').agg(repeated_revenue_usd=('Line Revenue (USD)', 'sum'), repeated_gross_profit_usd=('Line Gross Profit (USD)', 'sum'))
branch_sensitivity = primary_branch.join(removed_branch).fillna(0)
branch_sensitivity['gross_profit_reduction_percent'] = 100 * branch_sensitivity['repeated_gross_profit_usd'] / branch_sensitivity['gross_profit_usd']
print('Financial sensitivity by branch:')
print(branch_sensitivity.round(3).to_string())


Extra identical rows: 132
All rows involved in exact repeats: 264
Affected purchases: 132
                  Keep all rows (primary)  Remove exact repeats (scenario)  Scenario minus primary
Product lines                   51,177.00                        51,045.00                 -132.00
Purchases                       33,887.00                        33,887.00                    0.00
Units                           54,087.00                        53,955.00                 -132.00
Revenue USD                    305,925.14                       305,183.71                 -741.43
Gross profit USD               179,430.49                       178,994.28                 -436.21
Removing exact repeats reduces total gross profit by $436.21 (0.243%).


Financial sensitivity by branch:
         revenue_usd  gross_profit_usd  repeated_revenue_usd  repeated_gross_profit_usd  gross_profit_reduction_percent
Branch                                                                                                                 
Garden      99748.55          46794.49                218.00                      98.33                           0.210
Harbour    150483.58          97753.94                384.53                     250.48                           0.256
Uptown      55693.01          34882.06                138.90                      87.40                           0.251


## 7. Check time coverage and product definitions

Having at least one purchase on every observed branch-date is a useful coverage check. It cannot establish that every real purchase was exported. A constant unit cost may be a fixed accounting value; it is not evidence that actual operating costs never change.


In [7]:
date_min = df['parsed_date'].min()
date_max = df['parsed_date'].max()
calendar_days = pd.date_range(date_min, date_max, freq='D')
branches = sorted(df['Branch'].unique().tolist())
expected_branch_days = pd.MultiIndex.from_product([branches, calendar_days], names=['Branch', 'parsed_date'])
branch_day_counts = df.groupby(['Branch', 'parsed_date']).size().reindex(expected_branch_days, fill_value=0)
missing_branch_days = int((branch_day_counts == 0).sum())

categories_per_product = df.groupby('Product')['Category'].nunique()
prices_and_costs = df.groupby(['Branch', 'Product'])[['Unit Price (USD)', 'Unit Cost (USD)']].nunique()
products_multiple_categories = int((categories_per_product > 1).sum())
constant_prices = int((prices_and_costs['Unit Price (USD)'] == 1).sum())
constant_costs = int((prices_and_costs['Unit Cost (USD)'] == 1).sum())

print('Date coverage:', date_min.date(), 'to', date_max.date())
print('Calendar days:', len(calendar_days))
print('Branches:', ', '.join(branches))
print('Products:', df['Product'].nunique(), '| Categories:', df['Category'].nunique())
print('Branch-days with no recorded purchase:', missing_branch_days, 'of', len(expected_branch_days))
print('Products assigned to more than one category:', products_multiple_categories)
print('Branch-product combinations:', len(prices_and_costs))
print('Combinations with one recorded list price throughout:', constant_prices)
print('Combinations with one recorded unit cost throughout:', constant_costs)
print('The file does not provide a direct before/after change in list prices within the same branch-product combination.')


Date coverage: 2026-01-01 to 2026-06-30
Calendar days: 181
Branches: Garden, Harbour, Uptown
Products: 45 | Categories: 9
Branch-days with no recorded purchase: 0 of 543
Products assigned to more than one category: 0
Branch-product combinations: 135
Combinations with one recorded list price throughout: 135
Combinations with one recorded unit cost throughout: 135
The file does not provide a direct before/after change in list prices within the same branch-product combination.


## 8. Record the audit and our decisions

These reports make the checks inspectable without rerunning the notebook. The source CSV is never overwritten. The reports contain aggregate findings, not individual customer records.

Continue with branch/product analysis while carrying these limitations forward:

- Retain exact repeats in the primary analysis and test important results without them.
- Keep missing customer IDs missing; use identified customers only for explicitly qualified customer analyses.
- Treat recorded unit costs as supplied accounting values until their definition can be verified.
- Keep gross profit distinct from net profit.
- A complete date grid does not guarantee a complete transaction export.
- Observed patterns and accounting consistency do not prove causation or future savings.


In [8]:
assert hashlib.sha256(data_path.read_bytes()).hexdigest() == source_hash, 'The source file changed during the audit.'
report_dir = project_root / 'reports'
report_dir.mkdir(exist_ok=True)

audit = {
    'scope': 'Step 1 data audit only; no predictive model or causal business recommendation.',
    'source_filename': data_path.name,
    'source_sha256': source_hash,
    'source_size_bytes': data_path.stat().st_size,
    'source_unchanged': True,
    'rows': len(raw),
    'columns': len(raw.columns),
    'purchases': len(transactions),
    'branches': branches,
    'products': int(df['Product'].nunique()),
    'categories': int(df['Category'].nunique()),
    'date_start': str(date_min.date()),
    'date_end': str(date_max.date()),
    'calendar_days': len(calendar_days),
    'branch_days_without_records': missing_branch_days,
    'missing_counts': {key: int(value) for key, value in raw.isna().sum().items()},
    'missing_customer_purchases': missing_customer_purchases,
    'missing_customer_purchase_percent': 100 * missing_customer_purchases / len(transactions),
    'distinct_recorded_customer_ids': int(raw['Customer ID'].nunique()),
    'failed_check_counts': checks,
    'text_whitespace_counts': text_issues,
    'money_check_tolerance_usd': 0.01,
    'max_revenue_error_usd': float(revenue_error.max()),
    'max_gross_profit_error_usd': float(profit_error.max()),
    'negative_revenue_rows': int((df['Line Revenue (USD)'] < 0).sum()),
    'negative_gross_profit_rows': int((df['Line Gross Profit (USD)'] < 0).sum()),
    'extra_exact_repeat_rows': repeat_count,
    'all_rows_involved_in_exact_repeats': int(affected_mask.sum()),
    'purchases_with_exact_repeats': repeat_transactions,
    'primary_totals': primary_totals,
    'deduplicated_scenario_totals': sensitivity_totals,
    'duplicate_gross_profit_usd': duplicate_profit,
    'duplicate_gross_profit_percent': duplicate_profit_percent,
    'duplicate_sensitivity_by_branch': branch_sensitivity.round(6).reset_index().to_dict(orient='records'),
    'products_with_multiple_categories': products_multiple_categories,
    'branch_product_combinations': len(prices_and_costs),
    'branch_products_with_constant_price': constant_prices,
    'branch_products_with_constant_cost': constant_costs,
    'treatment': {
        'original_file': 'Unchanged; no rows dropped and no missing IDs filled.',
        'duplicate_rows': 'Retain in primary analysis; remove exact repeats only in a sensitivity scenario.',
        'customer_ids': 'Leave missing; identified-customer conclusions may not generalize to all purchases.',
        'working_copy': 'Parse dates/times and numeric values without overwriting the source.',
    },
    'runtime': {'python': platform.python_version(), 'pandas': pd.__version__},
}
(report_dir / 'step1_audit.json').write_text(json.dumps(audit, indent=2, allow_nan=False) + '\n', encoding='utf-8')

report = f"""# Stories Coffee: Step 1 data audit

The file passes the structural and arithmetic checks performed. It is usable for exploratory branch and product analysis with the limitations below. These checks do not verify the completeness of the export or the economic definition of its costs.

## Scope and integrity

- {len(raw):,} product lines, {len(transactions):,} purchases, {df['Product'].nunique()} products, and {len(branches)} branches.
- Dates: {date_min.date()} through {date_max.date()} ({len(calendar_days)} calendar days).
- SHA-256: `{source_hash}`.
- Source CSV unchanged. All rows are retained in the primary analysis.
- Python {platform.python_version()}, pandas {pd.__version__}.

## Results

| Check | Result |
|---|---|
| Required fields other than Customer ID | No missing values |
| Dates, timestamps, numeric fields, quantity and discount validity | No failures in the checks performed |
| Revenue identity | {checks['Revenue mismatches over one cent']} discrepancies above $0.01 per line |
| Gross-profit identity | {checks['Gross-profit mismatches over one cent']} discrepancies above $0.01 per line |
| Conflicting date/time/branch/order/customer within a transaction | {checks['Transactions with any metadata conflict']} transactions |
| Branch-days with no recorded purchase | {missing_branch_days} of {len(expected_branch_days)} |
| Extra identical rows | {repeat_count}, affecting {repeat_transactions} purchases |
| Missing Customer ID | {raw['Customer ID'].isna().sum():,} lines; {missing_customer_purchases:,} purchases ({missing_customer_purchases / len(transactions):.2%}) |
| Recorded customer IDs | {raw['Customer ID'].nunique():,}; this does not count unidentified customers |
| Branch-product pairs with constant recorded price and cost | {constant_prices} constant-price pairs and {constant_costs} constant-cost pairs, out of {len(prices_and_costs)} |

## Repeated-row sensitivity

The schema has no unique line-item ID. Exact repeats could be duplicate exports or legitimate repeated items. Neither explanation is proven.

| Measure | Keep all rows (primary) | Remove exact repeats (scenario) |
|---|---:|---:|
| Revenue | ${primary_totals['Revenue USD']:,.2f} | ${sensitivity_totals['Revenue USD']:,.2f} |
| Gross profit | ${primary_totals['Gross profit USD']:,.2f} | ${sensitivity_totals['Gross profit USD']:,.2f} |
| Purchases | {primary_totals['Purchases']:,} | {sensitivity_totals['Purchases']:,} |

Removing repeats reduces revenue by ${duplicate_revenue:,.2f} and gross profit by ${duplicate_profit:,.2f} ({duplicate_profit_percent:.3f}% of primary gross profit). This is a sensitivity calculation, not lost profit or recoverable savings. A small overall effect does not guarantee a small effect in every subgroup.

## Decisions and limitations

1. Keep original rows. Recheck the final business finding under the deduplicated scenario.
2. Leave missing Customer IDs missing. Do not group all unidentified purchases into one customer or assume this group behaves like identified customers.
3. Use transaction-level aggregates for basket questions; do not treat every product line as an independent purchase.
4. Treat constant recorded unit costs as accounting inputs until their meaning is verified. Passing arithmetic checks does not validate their business interpretation.
5. Report gross profit, not net profit. Rent, wages, and other operating costs are not separately available.
6. Presence on all branch-dates does not prove all sales were included. Customer-ID consistency across visits also remains an assumption.
7. Observational comparisons cannot by themselves prove that discounts, prices, or branches cause a result.

Next: investigate a focused business question and carry these limitations into its validation.
"""
(report_dir / 'step1_audit.md').write_text(report, encoding='utf-8')
print('Audit completed. Original CSV unchanged.')
print('Reports saved to reports/step1_audit.json and reports/step1_audit.md')
print('Working dataframes available for the next step: df (product lines), transactions (purchases).')


Audit completed. Original CSV unchanged.
Reports saved to reports/step1_audit.json and reports/step1_audit.md
Working dataframes available for the next step: df (product lines), transactions (purchases).
